In [2]:
from sshtunnel import SSHTunnelForwarder
import mysql.connector
import os
from dotenv import load_dotenv
from pprint import pprint
# Load .env
load_dotenv()

# Config
ssh_host = os.getenv('SSH_HOST')
ssh_user = os.getenv('PYTHONANYWHERE_USERNAME')
ssh_password = os.getenv('PYTHONANYWHERE_PASSWORD')
db_password = os.getenv('PYTHONANYWHERE_DB_PASSWORD')
db_name = os.getenv('PYTHONANYWHERE_DB_NAME')
db_host_remote = f'{ssh_user}.mysql.pythonanywhere-services.com'

# Start SSH tunnel
tunnel = SSHTunnelForwarder(
    (ssh_host, 22),
    ssh_username=ssh_user,
    ssh_password=ssh_password,
    remote_bind_address=(db_host_remote, 3306)
)
tunnel.start()
print("🔐 SSH tunnel established.")

# Connect to MySQL
conn = mysql.connector.connect(
    user=ssh_user,
    password=db_password,
    host='127.0.0.1',
    port=tunnel.local_bind_port,
    database=db_name,
    use_pure=True
)
cursor = conn.cursor()
print("✅ MySQL connection is open.")


🔐 SSH tunnel established.
✅ MySQL connection is open.


In [3]:
import pandas as pd

def fetch_table_as_dataframe(table_name, conn):
    """
    Fetch all rows from a given table as a pandas DataFrame.
    Args:
        table_name (str): The name of the table to fetch.
        conn: The active MySQL connection object.
    Returns:
        pd.DataFrame: DataFrame containing all rows from the table.
    """
    cursor = conn.cursor(dictionary=True)
    cursor.execute(f"SELECT * FROM {table_name}")
    rows = cursor.fetchall()
    return pd.DataFrame(rows)

def get_all_tables_as_dataframes(conn):
    """
    Fetch all main tables as DataFrames in a dictionary.
    Args:
        conn: The active MySQL connection object.
    Returns:
        dict: Dictionary with table names as keys and DataFrames as values.
    """
    tables = ['pdfs', 'metadata', 'text_files']
    return {table: fetch_table_as_dataframe(table, conn) for table in tables}

def get_last_uploaded_files(df, limit=5):
    """
    Get the last uploaded files from a DataFrame, sorted by 'id' descending.
    Args:
        df (pd.DataFrame): DataFrame to sort.
        limit (int): Number of rows to return.
    Returns:
        pd.DataFrame: The last 'limit' rows by 'id'.
    """
    return df.sort_values(by='id', ascending=False).head(limit)

def get_data_health_report(df):
    """
    Generate a health report for a DataFrame, including nulls, uniqueness, and duplicates.
    Args:
        df (pd.DataFrame): DataFrame to analyze.
    Returns:
        dict: Health report for each column.
    """
    report = {}
    for col in df.columns:
        report[col] = {
            'null_count': df[col].isnull().sum(),
            'unique_count': df[col].nunique(),
            'is_unique': df[col].is_unique,
            'duplicates': df.duplicated(subset=col).sum()
        }
    return report

In [ ]:
# Example usage in your notebook:
tables = get_all_tables_as_dataframes(conn)
pdfs_df = tables['pdfs']
last_files = get_last_uploaded_files(pdfs_df, limit=5)
health_report = get_data_health_report(pdfs_df)
print(health_report)

In [5]:
pprint(health_report)

{'file_path': {'duplicates': 0,
               'is_unique': True,
               'null_count': 0,
               'unique_count': 318},
 'filename': {'duplicates': 0,
              'is_unique': True,
              'null_count': 0,
              'unique_count': 318},
 'id': {'duplicates': 0,
        'is_unique': True,
        'null_count': 0,
        'unique_count': 318}}


In [6]:
# Generate and display health reports for all tables
for table_name, df in tables.items():
    print(f"\n=== Health Report for '{table_name}' ===")
    report = get_data_health_report(df)
    from pprint import pprint
    pprint(report)
    # Or, for tabular view:
    display(pd.DataFrame(report).T)


=== Health Report for 'pdfs' ===
{'file_path': {'duplicates': 0,
               'is_unique': True,
               'null_count': 0,
               'unique_count': 318},
 'filename': {'duplicates': 0,
              'is_unique': True,
              'null_count': 0,
              'unique_count': 318},
 'id': {'duplicates': 0,
        'is_unique': True,
        'null_count': 0,
        'unique_count': 318}}


,null_count,unique_count,is_unique,duplicates
id,0,318,True,0
filename,0,318,True,0
file_path,0,318,True,0



=== Health Report for 'metadata' ===
{'action_taken': {'duplicates': 309,
                  'is_unique': False,
                  'null_count': 42,
                  'unique_count': 8},
 'address': {'duplicates': 199,
             'is_unique': False,
             'null_count': 0,
             'unique_count': 119},
 'case_number': {'duplicates': 6,
                 'is_unique': False,
                 'null_count': 0,
                 'unique_count': 312},
 'cdcr_number': {'duplicates': 3,
                 'is_unique': False,
                 'null_count': 0,
                 'unique_count': 315},
 'cohort': {'duplicates': 313,
            'is_unique': False,
            'null_count': 0,
            'unique_count': 5},
 'completion_date': {'duplicates': 150,
                     'is_unique': False,
                     'null_count': 20,
                     'unique_count': 167},
 'convict_name': {'duplicates': 3,
                  'is_unique': False,
                  'null_count': 0,


,null_count,unique_count,is_unique,duplicates
id,0,318,True,0
pdf_id,0,318,True,0
date_stamped,21,87,False,230
judge,0,244,False,74
county,0,38,False,280
address,0,119,False,199
convict_name,0,315,False,3
cdcr_number,0,315,False,3
case_number,0,312,False,6
sentence_date,0,304,False,14



=== Health Report for 'text_files' ===
{}


""


In [9]:
# For pdfs table
last_pdf = pdfs_df.sort_values(by='id', ascending=False).iloc[0]
print("Most recent PDF record (by ID):")
print(last_pdf)

# For metadata table
last_meta = meta_df.sort_values(by='id', ascending=False).iloc[0]
print("\nMost recent metadata record (by ID):")
print(last_meta)

Most recent PDF record (by ID):
id                                                         987
filename                          corrected_GE_Lynn_T50359.pdf
file_path    /home/RSCAP/shared/archive_directory/corrected...
Name: 317, dtype: object

Most recent metadata record (by ID):
id                                                                    938
pdf_id                                                                987
date_stamped                                             August  10, 2022
judge                                                      Susan E. Green
county                                                             Sutter
address                      1175 Civic Center Blvd., Yuba City, CA 95993
convict_name                                                  Steven Lynn
cdcr_number                                                        T50359
case_number                                                    CRF01-2459
sentence_date                                     